In [52]:
from pathlib import Path
import numpy as np
from PIL import Image
import pandas as pd

# ===================== 需要你修改的路径 =====================

# 多个方法生成结果的文件夹
GENERATED_DIRS = {
    "PaletteGAN": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\pix2pix_code04\generated_results_dc_hed_contour_clean_"),
    # 如果之后还要算其他模型，继续往下加
    "FlexIcon-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\FlexIcon\test_output\batch_fake"),
    "IconFlow-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\IconFlow\test_results\generated"),
    "Comicolorization-adapted": Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\Comicolorization\output"),
}

# 调色板 npy 文件夹
PALETTE_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\dataset\color_palette")

# 掩码图片文件夹
MASK_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\dataset\mask")

# 输出结果文件夹
OUTPUT_DIR = Path(r"C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\palette_metric_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 掩码阈值
# 掩码像素值 > MASK_THRESHOLD 的区域会被认为是服装主体
MASK_THRESHOLD = 127

# 支持的图片格式
IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
MASK_EXTS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

In [53]:
def rgb_to_lab(rgb):
    """
    将 RGB 转换到 CIELAB
    输入:
        rgb: numpy array, shape (..., 3), 值域 [0, 1]
    输出:
        lab: numpy array, shape (..., 3)
    """
    rgb = np.asarray(rgb, dtype=np.float64)
    rgb = np.clip(rgb, 0.0, 1.0)

    # sRGB gamma correction
    mask = rgb > 0.04045
    rgb_linear = np.empty_like(rgb)
    rgb_linear[mask] = ((rgb[mask] + 0.055) / 1.055) ** 2.4
    rgb_linear[~mask] = rgb[~mask] / 12.92

    # sRGB -> XYZ (D65)
    matrix = np.array([
        [0.4124564, 0.3575761, 0.1804375],
        [0.2126729, 0.7151522, 0.0721750],
        [0.0193339, 0.1191920, 0.9503041]
    ])

    xyz = rgb_linear @ matrix.T
    xyz = xyz * 100.0

    # D65 reference white
    white = np.array([95.047, 100.000, 108.883])
    xyz_scaled = xyz / white

    epsilon = 0.008856
    kappa = 903.3

    f = np.where(
        xyz_scaled > epsilon,
        np.cbrt(xyz_scaled),
        (kappa * xyz_scaled + 16) / 116
    )

    L = 116 * f[..., 1] - 16
    a = 500 * (f[..., 0] - f[..., 1])
    b = 200 * (f[..., 1] - f[..., 2])

    lab = np.stack([L, a, b], axis=-1)
    return lab

In [54]:
def normalize_palette_rgb(palette):
    """
    将调色板统一转换到 RGB [0, 1]
    支持：
    1. [0, 255]
    2. [0, 1]
    3. [-1, 1]
    """
    palette = np.asarray(palette, dtype=np.float64)

    if palette.shape != (6, 3):
        raise ValueError(f"Palette shape should be (6, 3), but got {palette.shape}")

    p_min = palette.min()
    p_max = palette.max()

    # [-1, 1]
    if p_min < 0 and p_max <= 1:
        palette = (palette + 1.0) / 2.0
    # [0, 1]
    elif p_max <= 1.0:
        palette = palette
    # [0, 255]
    else:
        palette = palette / 255.0

    return np.clip(palette, 0.0, 1.0)


def load_image_rgb(image_path):
    """
    读取生成图像为 RGB [0,1]
    """
    img = Image.open(image_path).convert("RGB")
    rgb = np.asarray(img).astype(np.float64) / 255.0
    return rgb


def load_mask(mask_path, target_size=None, threshold=127):
    """
    读取掩码图并转为布尔 mask
    输入:
        mask_path: 掩码路径
        target_size: (W, H)，如果提供且掩码大小不一致，则最近邻缩放
        threshold: 二值阈值
    输出:
        mask: bool array, shape (H, W)
    """
    mask_img = Image.open(mask_path).convert("L")

    if target_size is not None and mask_img.size != target_size:
        mask_img = mask_img.resize(target_size, Image.NEAREST)

    mask_arr = np.asarray(mask_img)
    mask = mask_arr > threshold
    return mask

In [55]:
def find_matching_file(folder, stem, exts):
    """
    在 folder 中查找与 stem 同名、后缀属于 exts 的文件
    例如 stem='0001'，可能找到 0001.png 或 0001.jpg
    """
    for ext in exts:
        path = folder / f"{stem}{ext}"
        if path.exists():
            return path
    return None


def find_images(folder):
    image_paths = []
    for ext in IMAGE_EXTS:
        image_paths.extend(folder.glob(f"*{ext}"))
    return sorted(image_paths)

In [56]:
def compute_palette_adherence_distance(image_path, palette_path, mask_path, mask_threshold=127):
    """
    计算单张生成图像在服装主体区域上的调色板一致性距离
    只在 mask 指定的服装主体区域上计算

    返回:
        score: 平均最近 CIELAB 色差，越低越好
        num_pixels: 参与计算的服装主体像素数量
    """
    # 读取生成图像
    rgb = load_image_rgb(image_path)
    h, w = rgb.shape[:2]

    # 读取掩码
    mask = load_mask(mask_path, target_size=(w, h), threshold=mask_threshold)

    # 读取调色板
    palette = np.load(palette_path)
    palette_rgb = normalize_palette_rgb(palette)

    # 只保留服装主体区域像素
    fg_pixels_rgb = rgb[mask]

    if fg_pixels_rgb.shape[0] == 0:
        raise ValueError(f"No garment pixels found in mask: {mask_path}")

    # 转为 CIELAB
    fg_pixels_lab = rgb_to_lab(fg_pixels_rgb)
    palette_lab = rgb_to_lab(palette_rgb)

    # 计算每个服装像素到 6 个调色板颜色的距离
    # shape: (num_pixels, 6)
    distances = np.linalg.norm(
        fg_pixels_lab[:, None, :] - palette_lab[None, :, :],
        axis=-1
    )

    # 每个像素取最近的调色板颜色距离
    min_distances = distances.min(axis=1)

    # 取平均
    score = float(min_distances.mean())
    num_pixels = int(fg_pixels_rgb.shape[0])

    return score, num_pixels

In [57]:
all_rows = []
missing_files = []

for method_name, generated_dir in GENERATED_DIRS.items():
    generated_dir = Path(generated_dir)

    if not generated_dir.exists():
        raise FileNotFoundError(f"Generated image folder not found: {generated_dir}")

    image_paths = find_images(generated_dir)

    if len(image_paths) == 0:
        raise FileNotFoundError(f"No images found in folder: {generated_dir}")

    print(f"\nProcessing method: {method_name}")
    print(f"Number of generated images: {len(image_paths)}")

    for image_path in image_paths:
        stem = image_path.stem

        palette_path = PALETTE_DIR / f"{stem}.npy"
        mask_path = find_matching_file(MASK_DIR, stem, MASK_EXTS)

        if not palette_path.exists():
            missing_files.append({
                "method": method_name,
                "image_name": image_path.name,
                "missing_type": "palette",
                "expected_path": str(palette_path)
            })
            continue

        if mask_path is None:
            missing_files.append({
                "method": method_name,
                "image_name": image_path.name,
                "missing_type": "mask",
                "expected_path": str(MASK_DIR / f"{stem}.*")
            })
            continue

        try:
            score, num_pixels = compute_palette_adherence_distance(
                image_path=image_path,
                palette_path=palette_path,
                mask_path=mask_path,
                mask_threshold=MASK_THRESHOLD
            )

            all_rows.append({
                "method": method_name,
                "image_name": image_path.name,
                "palette_name": palette_path.name,
                "mask_name": mask_path.name,
                "palette_distance": score,
                "garment_pixels": num_pixels
            })

        except Exception as e:
            print(f"Error processing {image_path.name}: {e}")

results_df = pd.DataFrame(all_rows)

if len(results_df) == 0:
    raise RuntimeError("No valid image-palette-mask triplets were processed.")

results_df.head()


Processing method: PaletteGAN
Number of generated images: 10076

Processing method: FlexIcon-adapted
Number of generated images: 10076

Processing method: IconFlow-adapted
Number of generated images: 10076

Processing method: Comicolorization-adapted
Number of generated images: 10076


,method,image_name,palette_name,mask_name,palette_distance,garment_pixels
0,PaletteGAN,000001.jpg,000001.npy,000001.png,12.115718,21857
1,PaletteGAN,000002.jpg,000002.npy,000002.png,12.768002,21841
2,PaletteGAN,000003.jpg,000003.npy,000003.png,15.805467,21808
3,PaletteGAN,000004.jpg,000004.npy,000004.png,11.738138,21804
4,PaletteGAN,000005.jpg,000005.npy,000005.png,10.760460,21792


In [58]:
summary_df = (
    results_df
    .groupby("method")
    .agg(
        num_images=("image_name", "count"),
        palette_distance_mean=("palette_distance", "mean"),
        palette_distance_std=("palette_distance", "std"),
        palette_distance_median=("palette_distance", "median"),
        palette_distance_min=("palette_distance", "min"),
        palette_distance_max=("palette_distance", "max")
    )
    .reset_index()
)

summary_df

,method,num_images,palette_distance_mean,palette_distance_std,palette_distance_median,palette_distance_min,palette_distance_max
0,Comicolorization-adapted,10076,14.036655,4.474004,13.349213,2.932279,57.545824
1,FlexIcon-adapted,10076,13.029887,3.589585,12.722182,3.069417,47.946402
2,IconFlow-adapted,10076,12.946492,3.621569,12.589054,2.952012,43.689694
3,PaletteGAN,10076,12.909985,3.547774,12.582043,3.069417,47.946402


In [59]:
per_image_csv = OUTPUT_DIR / "palette_adherence_per_image.csv"
summary_csv = OUTPUT_DIR / "palette_adherence_summary.csv"

results_df.to_csv(per_image_csv, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print(f"Per-image results saved to: {per_image_csv}")
print(f"Summary results saved to: {summary_csv}")

if len(missing_files) > 0:
    missing_df = pd.DataFrame(missing_files)
    missing_csv = OUTPUT_DIR / "missing_files.csv"
    missing_df.to_csv(missing_csv, index=False, encoding="utf-8-sig")
    print(f"Missing file list saved to: {missing_csv}")
else:
    print("No missing files.")

Per-image results saved to: C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\palette_metric_results\palette_adherence_per_image.csv
Summary results saved to: C:\Users\35278\Desktop\fashion_coloring_file\pycharm_file\CGAN_Sketch_coloring\evaluation_methods\palette_metric_results\palette_adherence_summary.csv
No missing files.
